# 03 — Backtesting a Factor Strategy

End-to-end backtest: factor scores → portfolio weights → NAV → tearsheet.

In [ ]:
import pandas as pd
from hailmary.data.providers import YahooFinanceProvider
from hailmary.models.factors import MomentumFactor, VolatilityFactor
from hailmary.models.multi_factor import MultiFactorModel
from hailmary.models.portfolio import FactorPortfolio
from hailmary.backtest.engine import BacktestEngine, Strategy
from hailmary.backtest.execution import ExecutionModel, FixedSlippage, PercentCommission
from hailmary.analytics.metrics import PerformanceMetrics
from hailmary.viz.performance_charts import PerformanceCharts

In [ ]:
yahoo = YahooFinanceProvider()
universe = ['AAPL','MSFT','GOOGL','AMZN','META','NVDA','TSLA','JPM','V','UNH',
            'XOM','JNJ','PG','MA','HD','BAC','PFE','ABBV','LLY','MRK']

bars = yahoo.get_bars(universe, start='2019-01-01', end='2024-01-01')
close = bars['close'].unstack(level=0).dropna(how='all')

# Benchmark: SPY
spy = yahoo.get_bars(['SPY'], start='2019-01-01', end='2024-01-01')
benchmark = spy['close'].unstack(level=0)['SPY']

In [ ]:
# Define strategy
class MomentumQualityStrategy(Strategy):
    def __init__(self):
        self.model = MultiFactorModel([
            MomentumFactor(lookback=252, skip=21),
            VolatilityFactor(window=63),
        ])
        self.portfolio_builder = FactorPortfolio(
            n_quantiles=5, construction='quantile', long_short=False, max_weight=0.10
        )

    def generate_weights(self, prices, timestamp, portfolio, **ctx):
        if len(prices) < 280:
            return pd.Series(dtype=float)
        scores = self.model.score(prices)
        return self.portfolio_builder.get_weights(scores)

In [ ]:
execution = ExecutionModel(
    slippage=FixedSlippage(bps=5),
    commissions=PercentCommission(pct=0.001),
)

engine = BacktestEngine(
    prices=close,
    strategy=MomentumQualityStrategy(),
    initial_capital=1_000_000,
    rebalance_frequency='ME',
    execution=execution,
    benchmark=benchmark,
    risk_free_rate=0.05,
)

result = engine.run(start='2020-01-01', end='2023-12-31')

In [ ]:
# Performance summary
metrics = PerformanceMetrics(result.returns, risk_free_rate=0.05)
bm_returns = benchmark.pct_change().dropna()
print(metrics.summary(bm_returns))

In [ ]:
charts = PerformanceCharts(result, name='Momentum + Low-Vol')
charts.tearsheet().show()

In [ ]:
charts.monthly_returns_heatmap().show()